# Final Data Preparation (Datasets merging and future engeneering)

#### Currently the project data is seperated into miltiple datasets, each with its own specific data. What this notebook will do, is that it will merge the datasets by structuring them into ones, with organaized and detailly selected features.The main aim of the notebook is to combine all of the datasets that the ptoject has cleaned, fix any problems if there are and prepare the data for modeling!

#### Data sources which have been used:
- **Football-Data.co.uk**
- **Understat data**
- **Soccerdata**
- **Transfermarkt data**
- **Elo ratings data**

#### Process of work:
**1.** First, the notebook will merge all of the datasets together \
**2.** Second, after the datasets are merged, the features will be handled, by either removing some repetative or useless ones! \
**3.** Third, after the data has been merged, structured and clraned, one final feature engineering will be made.There are still some more features to be added, which are extremely important and for their implementation were needed all of the datasets together! \
**4.** Fourth, when the data is merged and the features are structured and organized, some surface cleaning will be made.For example optimizing the final dataset memory usage or fixing any data types - some little cleaning! \
**5.** Finally the processed dataset will be saved as the final dataset of the project, ready for modeling!

#### About the modelling:
I want to specify that this notebook will not make any imputation, encoding or scaling.These things will be left for another notebook which will be made specifically to prepara the data for modeling!

### About the final features which will be created:
Here are the features that are planned to be created:

#### Match level features:
##### Rolling features(Rolling form features are among the strongest predictors!): 
- rolling_xG_last_5
- rolling_xG_last_5
- rolling_xGA_last_5
- rolling_xGA_last_5
- rolling_shots_last_5
- rolling_sot_last_5

##### Form(Points gathred from matches)
- home_form_last_5
- away_form_last_5

##### Shot Aggregation data
- home_big_chances
- away_big_chances
- home_avg_xG_per_shot
- away_avg_xG_per_shot

##### Metrics differences:
- xG_diff
- goal_diff
- rolling_goal_diff
- rolling_xG_diff
- elo_rating_diff
- possesion_diff

##### Head-to-Head records
- h2h_record
- h2h_rolling_goal_diff_5
- h2h_rolling_xG_diff_5
- h2h_rolling_elo_diff_5

---

#### External features
- team's formations stability
- team's manager stability and consistency
- referee bias
- attendance pressure

These features will be explained in the process of work!

#### Getting started:

Now, as everything has been explained lets start the work:

### Loading the required libraries:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from football_betting_analysis.config import PROCESSED_DATA_PATH

from football_betting_analysis.data.data_cleaning import optimize_dataframe_memory
from football_betting_analysis.data.data_cleaning import validate_and_cast_dataframe_dtypes
from football_betting_analysis.data.match_resolver import resolve_date_shift_matches

from football_betting_analysis.features.features_creation import calculate_rolling_metric, \
    calculate_weighted_rolling_average, compute_formation_stability_features, compute_referee_bias_features, \
    compute_attendance_features, compute_manager_features

from football_betting_analysis.data.save_data_into_file import save_data

### Loading the datasets:

In [2]:
# Football-Data.co.uk data
football_data_df = pd.read_parquet('../../data/interim/football_data_co_uk/interim_matches_v1.parquet')

# Elo ratings data
elo_ratings_df = pd.read_parquet('../../data/interim/elo_ratings/interim_elo_ratings_v1.parquet')
elo_ratings_matches_df = pd.read_parquet('../../data/interim/elo_ratings/interim_elo_matches_v1.parquet')

# Understat data
understat_matches_df = pd.read_parquet('../../data/interim/understat_data/interim_matches_v1.parquet')
understat_matches_shots_df = pd.read_parquet('../../data/interim/understat_data/interim_matches_shots_v1.parquet')
understat_players_df = pd.read_parquet('../../data/interim/understat_data/interim_players_v1.parquet')
understat_teams_context_df = pd.read_parquet('../../data/interim/understat_data/interim_teams_context_v1.parquet')

# Soccer data
soccer_data_matches_df = pd.read_parquet('../../data/interim/soccer_data/interim_matches_v1.parquet')

# Transfermarkt data:
transfermarkt_matches_df = pd.read_parquet('../../data/interim/transfermarkt_data/interim_matches_v3.parquet')

### Prerequestins:

#### Storing datasets:
I want to specify that the final dataset will be stored into the `data/processed` directory, which directoty is for storing finaly processed and totally cleaned datasets which are ready for analyses and modeling!
---

### Initial cleaning

#### Merging requirements:
Before merging the datasets we should confirm three very important detailes: \
1.**STANDARDIZED COLUMN NAMES**:
- The datasets should have a unique key by which they to be merged!As the datasets are not collected from one source, they do not have predefined matches ids, which means that this key should be constructed from the features in the datasets.This unique key surely will be the combination between: **season, datetime, h_title, a_title** - In the cleaning phase, we have guaranteed that there is no a dataset which violates this unique criteria.Also the datasets are all designed to have these unique features named in a consistent way, which means that all of the datasets have them and we are freely to use them as a unique key!

2.**STANDARDIZED TEAM NAMES**:
- The next thing which is extrmely important and required for the datasets to be merged correctly, is for them to have consistent team names.This also is something which is already achived in the cleaning phase and currently all og the datasets have valid and consistent teams names

3.**STANDARDIZED DATETIME**:
- Another thing, is that the datasets should have their datetimes in one format, which should be the standard one: `YYYYYYYY-MM-DD` without any timestamps etc.This is something which is valid for the Elo Ratigns dataset and for the Football-Data.co.uk dataset, but the Understat dataset contains its datetimes with timestamps.So this should be fixed!

Everything from these requirements has been followed and totally made across all of the datasets!However, there are still some things that should be done before merging the datasets.

1.The Understant matches and shots datasets are with not normalized datetime columns.Currently they are in format: [Y-M-D:H:M:S].So these columns should be normalized to only [Y-M-D]. \
2.The Transfermarkt dataset has its datetime column with name: '**date**'.All of the other datasets are with datetime columns named: '**datetime**'.So the transfermarkt datetime column should be renamed! \
3.The soccer data also has its datetime column with name: '**date**, so this also should be fixed!

Except these two things, everything else is totally complied! \
Now lets fix these two things:

### Normalizing the understant datetime columns:

In [3]:
understat_matches_df['datetime'] = understat_matches_df['datetime'].dt.normalize()
understat_matches_shots_df['datetime'] = understat_matches_shots_df['datetime'].dt.normalize()

### Renaming the datetime column of the Transfermarkt data:

In [4]:
transfermarkt_matches_df = transfermarkt_matches_df.rename(columns={
    'date': 'datetime'
})

### Renaming the datetime column of the Soccer data:

In [5]:
soccer_data_matches_df = soccer_data_matches_df.rename(columns={
    'date': 'datetime'
})

Ok now everything is perfect!

---

### Creating the Base Match dataset:

I will start by constructing the first dataset, which will serve as a base.This dataset should contain the main match-level data.

To cretae the dataset I should use all of the five datasets.

- I will get the main match stats from the `Elo_Ratings matches dataset`, which dataset will also give me the elo ratings metrics which are important.

- I will use the Football-Data.co.uk dataset to get the matches betting odds, which in fact are also contained into the Elo Ratings datasets, but the Footbal-Data.co.uk is much more reliable and specifically created to provide betting odds!

- I will use the Understat dataset which will give me the **expected goals** metrics, which are already known to be extremely important!

- I will use the Soccerdata dataset to get the teams ball possesions and also the exact time of the matches.The ball posesion is an extremely important features because it identifies the dominating team in the match!

- I will use the Transfermarkt dataset to get all of the implemented player-influence and match importance features!

So lets begin.

In [6]:
matches_final_df = pd.merge(
    left=elo_ratings_matches_df, 
    right=understat_matches_df,
    on=['season', 'datetime', 'h_title', 'a_title'], 
    how='inner',
    indicator=True
)

Now lets see if everything has matched, by simply cheack if the shape of the dataset is equal to 4180, which is the total amount of matches for all of datasets!

In [7]:
matches_final_df.shape

(4111, 45)

Well, we can see that there is some problem with the matching, because there are **69** matches which does not match the unique criteria!

Lets see from where this might be coming!Where the matches differ? \
I will see that by using an outer join.

In [8]:
elo_ratings_matches_df = elo_ratings_matches_df.copy()
understat_matches_df = understat_matches_df.copy()

# Creating helpers
# I am doing that because, in order to fix the problem with the dates, I need to store the indeces of datasets
# so that later I can use them to locate the unmatching matches and fix them!
elo_ratings_matches_df["elo_rating_idx"] = elo_ratings_matches_df.index
understat_matches_df["understat_idx"] = understat_matches_df.index

In [9]:
exact_merge = elo_ratings_matches_df.merge(
    understat_matches_df,
    on=['season', 'datetime', 'h_title', 'a_title'], 
    how='outer',
    indicator=True,
    suffixes=("", "_u"),
)

Now lets take only the ones which didn't match:

In [10]:
left_only = exact_merge[exact_merge["_merge"] == "left_only"].copy()
right_only = exact_merge[exact_merge["_merge"] == "right_only"].copy()

#### As I have inspected the unmatching matches, I have identified the problem, which is simply caused from the the **discrepancies between the dates** of the different datasets.And more specifically the problem comes from the fact that certain matches from the **Understat data** are with incorrectly defined dates!This could have happen from problems while the scraping, or just inconsistencies between the time zones and many more other reasons.This is not important.What is important is that the problem is identified and it should be fixed.

So what I will do is that I will use an automated tolerant matching function, which will take the two unmatching extracts from the data frames, and will try to match the football games with small date inconsistencies - max 1 day, because otherwise the matches could be totally different, but it is just not possible, in a domestic league like La Liga, two teams to have played two matches two days in row - It is just not possible!

The function which I am going to use is defined at: [resolve_date_shift_matches](../../src/football_betting_analysis/data/match_resolver.py)

So lets apply the function:

In [11]:
resolved_matches = resolve_date_shift_matches(
    left_df=left_only,
    right_df=right_only,
    use_index=True,
    left_index_col='elo_rating_idx',
    right_index_col='understat_idx',
    max_days=1
)

In [12]:
resolved_matches

,left_index,right_index
0,114738.0,1403.0
1,114894.0,1406.0
2,114893.0,1407.0
3,115075.0,1412.0
4,115076.0,1413.0
...,...,...
64,140019.0,7915.0
65,141805.0,7967.0
66,142388.0,7987.0
67,196423.0,17503.0


Now as we have collected the unmatching matches in pairs, lets fix the Understat ones, by simply updating the date of the matches one day before!

In [13]:
for _, row in resolved_matches.iterrows():

    left_idx = row["left_index"]
    right_idx = row["right_index"]

    corrected_date = elo_ratings_matches_df.loc[left_idx, "datetime"]

    understat_matches_df.loc[right_idx, "datetime"] = corrected_date

And now lets remove the helping columns from the datasets!

In [14]:
elo_ratings_matches_df = elo_ratings_matches_df.drop(columns=['elo_rating_idx'])
understat_matches_df = understat_matches_df.drop(columns=['understat_idx'])

And now lets again make the merge, and see if we have fixed the problems with the dates:

In [15]:
test_df = pd.merge(
    left=elo_ratings_matches_df, 
    right=understat_matches_df,
    on=['season', 'datetime', 'h_title', 'a_title'], 
    how='inner'
)

In [16]:
test_df.shape

(4180, 44)

Yes everything is perfect!We are good to go!

Now before moving to the merges, I will define the base dataset which will serve as a starting point to the merges.This base dataset will be the Elo Ratings dataset, because it contain almost everything from the other datasets.

### Defining the base matches dataset:

In [17]:
base_matches_df = elo_ratings_matches_df.copy()

Now what I will do is that I will merge this dataset with **Understat** by selecting only the **specific columns** that I want to get from the Understat matches data, just because the only thing that I need from the understat data are the **expected goals metrics**, which I do not have in any other dataset!

So here is what I will get from understat:

In [18]:
UNDERSTAT_MATCH_FEATURES = [
    "season",
    "datetime",
    "h_title",
    "a_title",
    "xG_h",
    "xG_a",
]

In [19]:
base_matches_df = base_matches_df.merge(
    understat_matches_df[UNDERSTAT_MATCH_FEATURES],
    on=["season", "datetime", "h_title", "a_title"],
    how="left",
    validate="one_to_one"
)

In [20]:
base_matches_df.shape

(4180, 37)

### Checking for some duplicates and missing values:

In [21]:
base_matches_df[['xG_h', 'xG_a']].isna().any()

xG_h    False
xG_a    False
dtype: bool

In [22]:
assert (
    base_matches_df[
        ["season", "datetime", "h_title", "a_title"]
    ]
    .duplicated()
    .sum()
    == 0
)

#### Now lets also merge the Football-Data dataset!

From Football-Data.co.uk I will get only the odds metrics, so:

In [23]:
FOOTBALL_DATA_FEATURES = [
    "season",
    "datetime",
    "h_title",
    "a_title",
    "odds_bet365_home",
    "odds_bet365_draw",
    "odds_bet365_away"
]

In [24]:
base_matches_df = base_matches_df.merge(
    football_data_df[FOOTBALL_DATA_FEATURES],
    on=['season', 'datetime', 'h_title', 'a_title'],
    how='left',
    validate='one_to_one',
    suffixes=['_elo', '']
)

In [25]:
base_matches_df.shape

(4180, 40)

In [26]:
base_matches_df.columns

Index(['league_code', 'season', 'datetime', 'h_title', 'a_title', 'home_elo',
       'away_elo', 'home_form_last_3', 'home_form_last_5', 'away_form_last_3',
       'away_form_last_5', 'home_goals_full', 'away_goals_full', 'result_full',
       'home_goals_half', 'away_goals_half', 'result_half', 'home_shots',
       'away_shots', 'home_shots_on_target', 'away_shots_on_target',
       'home_fouls', 'away_fouls', 'home_corners', 'away_corners',
       'home_yellow_cards', 'away_yellow_cards', 'home_red_cards',
       'away_red_cards', 'odds_bet365_home_elo', 'odds_bet365_draw_elo',
       'odds_bet365_away_elo', 'handicap_size', 'handicap_home',
       'handicap_away', 'xG_h', 'xG_a', 'odds_bet365_home', 'odds_bet365_draw',
       'odds_bet365_away'],
      dtype='str')

As the elo rating dataset also have betting features I will drop them, and leave only the ones from the **Football-Data**.

In [27]:
base_matches_df = base_matches_df.drop(
    columns=[
        "odds_bet365_home_elo",
        "odds_bet365_draw_elo",
        "odds_bet365_away_elo"
    ]
)

### Merging the Soccer data:
From the soccer data I will get **the exact time of the matches, the team's ball possesion, the referee, the team's formations and the round of the matches**!

In [28]:
SOCCER_DATA_FEATURES = [
    "season",
    "datetime",
    "h_title",
    "a_title",
    "time",
    "h_poss",
    "a_poss",
    "referee",
    "h_formation",
    "a_formation",
    "round"
]

In [29]:
base_matches_df = base_matches_df.merge(
    soccer_data_matches_df[SOCCER_DATA_FEATURES],
    on=['season', 'datetime', 'h_title', 'a_title'],
    how='left',
    validate='one_to_one'
)

In [30]:
base_matches_df.shape

(4180, 44)

In [31]:
base_matches_df.columns

Index(['league_code', 'season', 'datetime', 'h_title', 'a_title', 'home_elo',
       'away_elo', 'home_form_last_3', 'home_form_last_5', 'away_form_last_3',
       'away_form_last_5', 'home_goals_full', 'away_goals_full', 'result_full',
       'home_goals_half', 'away_goals_half', 'result_half', 'home_shots',
       'away_shots', 'home_shots_on_target', 'away_shots_on_target',
       'home_fouls', 'away_fouls', 'home_corners', 'away_corners',
       'home_yellow_cards', 'away_yellow_cards', 'home_red_cards',
       'away_red_cards', 'handicap_size', 'handicap_home', 'handicap_away',
       'xG_h', 'xG_a', 'odds_bet365_home', 'odds_bet365_draw',
       'odds_bet365_away', 'time', 'h_poss', 'a_poss', 'referee',
       'h_formation', 'a_formation', 'round'],
      dtype='str')

### Merging the Tranfermarkt data
This is the biggest dataset of all and it contains a lot of columns.

Lets actually see them all and decide what should be taken:

In [32]:
transfermarkt_matches_df.columns.tolist()

['game_id',
 'competition_id',
 'season',
 'round',
 'datetime',
 'home_club_id',
 'away_club_id',
 'home_club_goals',
 'away_club_goals',
 'home_club_position',
 'away_club_position',
 'home_club_manager_name',
 'away_club_manager_name',
 'stadium',
 'attendance',
 'referee',
 'home_club_formation',
 'away_club_formation',
 'h_title',
 'a_title',
 'competition_type',
 'home_missing_players_count',
 'home_missing_key_players_count',
 'home_missing_star_players_count',
 'home_missing_importance_sum',
 'home_missing_defenders',
 'home_missing_midfielders',
 'home_missing_forwards',
 'home_starting_goalkeeper_missing',
 'home_missing_captain',
 'home_missing_top1_player',
 'home_missing_top3_player',
 'home_available_strength',
 'home_missing_expected_starter_strength',
 'home_missing_gk_strength',
 'home_missing_def_strength',
 'home_missing_mid_strength',
 'home_missing_fwd_strength',
 'home_missing_importance_ratio',
 'home_squad_stability',
 'away_missing_players_count',
 'away_missin

Ok these are the features that should be removed:

In [33]:
# Repetative and useless features:
FEATURES_TO_REMOVE = [
    'competition_id',
    'round',
    'home_club_goals',
    'away_club_goals',
    'referee',
    'competition_type',
]

In [34]:
TRANSFERMARKT_DATA_FEATURES = [col for col in transfermarkt_matches_df.columns if col not in FEATURES_TO_REMOVE]

In [35]:
len(TRANSFERMARKT_DATA_FEATURES)

122

Now before merging the data with the Transfermarkt data, we need to do something else!I have identified that there are two matches which differ in their dates.

Lets see them:

In [36]:
test = base_matches_df.merge(
    transfermarkt_matches_df[TRANSFERMARKT_DATA_FEATURES],
    on=['season', 'datetime', 'h_title', 'a_title'],
    how='outer',
    indicator=True,
    validate='one_to_one'
)

In [37]:
test.shape

(4182, 163)

Here are the mismatched matches:

In [38]:
test[test._merge != 'both']

,league_code,season,datetime,h_title,a_title,home_elo,away_elo,home_form_last_3,home_form_last_5,away_form_last_3,...,el_direct_clash,relegation_direct_clash,relegation_six_pointer,direct_position_clash,position_gap,points_gap_between_teams,cl_spots,el_spots,conf_spots,_merge
3180,SP1,2022/2023,2022-12-29,Atletico Madrid,Elche,1788.010010,1548.130005,1.0,5.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3183,NaN,2022/2023,2022-12-30,Atletico Madrid,Elche,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,16.0,20.0,4.0,2.0,1.0,right_only
3578,NaN,2023/2024,2023-12-10,Granada,Athletic Club,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,14.0,21.0,4.0,2.0,1.0,right_only
3579,SP1,2023/2024,2023-12-11,Granada,Athletic Club,1582.150024,1745.030029,1.0,1.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


As we can see the dates of the matches differ in a single day.What I will do is that I will fix these records using the right dates of the matches.For the match **Atletico Madrid vs. Elche**, the actual match date is 2022-12-30, not 29 and for the match **Granada vs. Athletic Club** the actual match date is 2023-12-11.

So as we can see there is one mistake in both of the datasets. \
I will fix the Transfermarkt mistake on the match: **Granada vs. Athletic Club** which is on **12.11 not 12.10** and also I will fix the **base_matches_df** mistake on the match: **Atletico Madrid vs. Elche** which is on **12.30 not 12.29!**

In [39]:
transfermarkt_matches_df.loc[
    (transfermarkt_matches_df.h_title == "Granada")
    & (transfermarkt_matches_df.a_title == "Athletic Club")
    & (transfermarkt_matches_df.datetime == "2023-12-10"),
    "datetime",
] = "2023-12-11"

base_matches_df.loc[
    (base_matches_df.h_title == "Atletico Madrid")
    & (base_matches_df.a_title == "Elche")
    & (base_matches_df.datetime == "2022-12-29"),
    "datetime"
] = "2022-12-30"

Now lets merge the datasets again and see if we have fixed the problem:

In [40]:
test = base_matches_df.merge(
    transfermarkt_matches_df[TRANSFERMARKT_DATA_FEATURES],
    on=['season', 'datetime', 'h_title', 'a_title'],
    how='outer',
    indicator=True,
    validate='one_to_one'
)

In [41]:
test.shape

(4180, 163)

Yes we are good to go:

In [42]:
base_matches_df = base_matches_df.merge(
    transfermarkt_matches_df[TRANSFERMARKT_DATA_FEATURES],
    on=['season', 'datetime', 'h_title', 'a_title'],
    how='left',
    validate='one_to_one'
)

In [43]:
base_matches_df.shape

(4180, 162)

In [44]:
base_matches_df.duplicated(subset='game_id').any()

np.False_

In [45]:
base_matches_df.columns.tolist()

['league_code',
 'season',
 'datetime',
 'h_title',
 'a_title',
 'home_elo',
 'away_elo',
 'home_form_last_3',
 'home_form_last_5',
 'away_form_last_3',
 'away_form_last_5',
 'home_goals_full',
 'away_goals_full',
 'result_full',
 'home_goals_half',
 'away_goals_half',
 'result_half',
 'home_shots',
 'away_shots',
 'home_shots_on_target',
 'away_shots_on_target',
 'home_fouls',
 'away_fouls',
 'home_corners',
 'away_corners',
 'home_yellow_cards',
 'away_yellow_cards',
 'home_red_cards',
 'away_red_cards',
 'handicap_size',
 'handicap_home',
 'handicap_away',
 'xG_h',
 'xG_a',
 'odds_bet365_home',
 'odds_bet365_draw',
 'odds_bet365_away',
 'time',
 'h_poss',
 'a_poss',
 'referee',
 'h_formation',
 'a_formation',
 'round',
 'game_id',
 'home_club_id',
 'away_club_id',
 'home_club_position',
 'away_club_position',
 'home_club_manager_name',
 'away_club_manager_name',
 'stadium',
 'attendance',
 'home_club_formation',
 'away_club_formation',
 'home_missing_players_count',
 'home_missing_k

### Dealing with the home/away team's formations:
I have left the team's formation form both Soccer data and Transfermarkt data **for reason**!In both datasets there are missing values in the formations features.However, fortunately, the datasets have missing values at different places and when we combine them, we will get all of the information!

I will use the pandas `combine_first` function which get the values from one of the features and fill them into the missing values places into the another features!

In [46]:
base_matches_df['home_formation'] = base_matches_df['home_club_formation'].combine_first(base_matches_df['h_formation'])
base_matches_df['away_formation'] = base_matches_df['away_club_formation'].combine_first(base_matches_df['a_formation'])

In [47]:
base_matches_df.away_formation.isna().any(), base_matches_df.home_formation.isna().any()

(np.False_, np.False_)

In [48]:
base_matches_df = base_matches_df.drop(columns=[
    'home_club_formation', 
    'away_club_formation',
    'h_formation',
    'a_formation'
])

In [49]:
base_matches_df.shape

(4180, 160)

## Feature engeneering

### Calculating Metrics Difference

I wll compute the goals, expected goals and elo_rating difference as features for each of the matches. \

In [50]:
base_matches_df['goal_diff'] = base_matches_df['home_goals_full'] - base_matches_df['away_goals_full']
base_matches_df['xG_diff'] = base_matches_df['xG_h'] - base_matches_df['xG_a']
base_matches_df['elo_rating_diff'] = base_matches_df['home_elo'] - base_matches_df['away_elo']
base_matches_df['poss_diff'] = base_matches_df['h_poss'] - base_matches_df['a_poss']

### Rolling averages:
Before adding new features, I should create a helping datasets! \
These helping datasets will store the matches in format: **team, opponent, venue(Home/Away)**!

This should be done, so that when calculating rolling features, the teams forms to be gathered from all their recent matches, regarding if they are the home or the away team!So what I will do is that I will take for each team, its home matches and its away matches, calculate their rolling form and then I will merge the helping dataset with the base one.

First I will get only the home teams:

In [51]:
home_df = base_matches_df[
    [
        "datetime", 
        "h_title", 
        "a_title", 
        "home_goals_full", 
        "away_goals_full", 
        "home_shots", 
        "away_shots", 
        "home_shots_on_target",
        "away_shots_on_target", 
        "xG_h", 
        "xG_a",
        "h_poss",
        "a_poss"
    ]
].rename(columns={
    "h_title": "team",
    "a_title": "opponent",
    "home_goals_full": "goals",
    "away_goals_full": "goals_against",
    "home_shots": "shots",
    "away_shots": "shots_against",
    "home_shots_on_target": "sot",
    "away_shots_on_target": "sot_against",
    "xG_h": "xG",
    "xG_a": "xGA",
    "h_poss": "poss",
    "a_poss": "poss_against"
})

# Calculating goals diff:
home_df["goal_diff"] = (
    home_df["goals"] -
    home_df["goals_against"]
)

# Calculating possesion diff:
home_df["poss_diff"] = (
    home_df["poss"] -
    home_df["poss_against"]
)

home_df["venue"] = "home"

In [52]:
away_df = base_matches_df[
    [
        "datetime",
        "a_title",
        "h_title",
        "away_goals_full",
        "home_goals_full",
        "away_shots",
        "home_shots",
        "away_shots_on_target",
        "home_shots_on_target",
        "xG_a",
        "xG_h",
        "a_poss",
        "h_poss"
    ]
].rename(columns={
    "a_title": "team",
    "h_title": "opponent",
    "away_goals_full": "goals",
    "home_goals_full": "goals_against",
    "away_shots": "shots",
    "home_shots": "shots_against",
    "away_shots_on_target": "sot",
    "home_shots_on_target": "sot_against",
    "xG_a": "xG",
    "xG_h": "xGA",
    "a_poss": "poss",
    "h_poss": "poss_against"
})

# Calculating goal diff:
away_df["goal_diff"] = (
    away_df["goals"] -
    away_df["goals_against"]
)

# Calculating possesion diff:
away_df["poss_diff"] = (
    away_df["poss"] -
    away_df["poss_against"]
)

away_df["venue"] = "away"

Now, as I have rightfully devided the teams as home and away ones, I will concat them:

In [53]:
team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [54]:
# Sorting the matches in chronological order
team_matches = team_matches.sort_values(
    ["team", "datetime", "venue"]
).reset_index(drop=True)

### Calculating rolling averages:

1.Expected goals:
- **home_rolling_xG_last_5, away_rolling_xG_last_5**
- **home_rolling_xGA_last_5, away_rolling_xGA_last_5**

2.Total Results
- **rolling_goals_last_5**
- **rolling_shots_last_5**, 
- **rolling_sot_last_5**
- **rolling_poss_last_5**

3.Form(Points gathred from matches)
- **home_form_last_5, away_form_last_5**

For the creation of these features I will use the `calculate_rolling_metric` function which, based on a given function and time window, will create a rolling metric of the provided column!

See the function at: [feature_creation](../../src/football_betting_analysis/features/features_creation.py)

Now lets use the function and create the rolling features:

In [55]:
rolling_features = [
    "xG",
    "xGA",
    "goals",
    "goals_against",
    "shots",
    "shots_against",
    "sot",
    "sot_against",
    "goal_diff",
    "poss_diff",
    "poss",
    "poss_against"
]

for feature in rolling_features:
    rolling_col = f"rolling_{feature}_5"
    
    team_matches[rolling_col] = calculate_rolling_metric(
        data=team_matches,
        group_col='team',
        column=feature,
        func='mean',
        window=5
    )

It is expected that **31** rows will be with **NaN** values, because in the dataset there are 31 **unique** teams, and each of their first matches has no **historical** data!

So lets see if they are actally 31 rows with NaN values:

In [56]:
team_matches['team'].nunique()

31

In [57]:
team_matches['team'].nunique() == len(team_matches[team_matches.isna().any(axis=1)])

False

Yes they are exactly 31!

##### Now lets again seperate the maches by home and away and prepare them for merge with the base matches dataset:

In [58]:
home_rolling = (
    team_matches[team_matches["venue"] == "home"]
    .rename(columns={
        "team": "h_title",
        "rolling_goals_5": "h_rolling_goals_5",
        "rolling_goals_against_5": "h_rolling_goals_against_5",
        "rolling_shots_5": "h_rolling_shots_5",
        "rolling_shots_against_5": "h_rolling_shots_against_5",
        "rolling_sot_5": "h_rolling_sot_5",
        "rolling_sot_against_5": "h_rolling_sot_against_5",
        "rolling_xG_5": "h_rolling_xG_5",
        "rolling_xGA_5": "h_rolling_xGA_5",
        "rolling_goal_diff_5": "h_rolling_goal_diff_5",
        "rolling_poss_diff_5": "h_rolling_poss_diff_5",
        "rolling_poss_5": "h_rolling_poss_5",
        "rolling_poss_against_5": "h_rolling_poss_against_5"
    })
)

home_rolling = home_rolling[
    [
        "datetime",
        "h_title",
        "h_rolling_goals_5",
        "h_rolling_goals_against_5",
        "h_rolling_shots_5",
        "h_rolling_shots_against_5",
        "h_rolling_sot_5",
        "h_rolling_sot_against_5",
        "h_rolling_xG_5",
        "h_rolling_xGA_5",
        "h_rolling_goal_diff_5",
        "h_rolling_poss_diff_5",
        "h_rolling_poss_5",
        "h_rolling_poss_against_5"
    ]
]

Now the away:

In [59]:
away_rolling = (
    team_matches[team_matches["venue"] == "away"]
    .rename(columns={
        "team": "a_title",
        "rolling_goals_5": "a_rolling_goals_5",
        "rolling_goals_against_5": "a_rolling_goals_against_5",
        "rolling_shots_5": "a_rolling_shots_5",
        "rolling_shots_against_5": "a_rolling_shots_against_5",
        "rolling_sot_5": "a_rolling_sot_5",
        "rolling_sot_against_5": "a_rolling_sot_against_5",
        "rolling_xG_5": "a_rolling_xG_5",
        "rolling_xGA_5": "a_rolling_xGA_5",
        "rolling_goal_diff_5": "a_rolling_goal_diff_5",
        "rolling_poss_diff_5": "a_rolling_poss_diff_5",
        "rolling_poss_5": "a_rolling_poss_5",
        "rolling_poss_against_5": "a_rolling_poss_against_5"
    })
)

away_rolling = away_rolling[
    [
        "datetime",
        "a_title",
        "a_rolling_goals_5",
        "a_rolling_goals_against_5",
        "a_rolling_shots_5",
        "a_rolling_shots_against_5",
        "a_rolling_sot_5",
        "a_rolling_sot_against_5",
        "a_rolling_xG_5",
        "a_rolling_xGA_5",
        "a_rolling_goal_diff_5",
        "a_rolling_poss_diff_5",
        "a_rolling_poss_5",
        "a_rolling_poss_against_5"
    ]
]

Now lets finaly merge the two helping datasets back to the original base matches dataset: 

In [60]:
base_matches_df = base_matches_df.merge(
    home_rolling,
    on=["datetime", "h_title"],
    how="left"
)

base_matches_df = base_matches_df.merge(
    away_rolling,
    on=["datetime", "a_title"],
    how="left"
)

In [61]:
base_matches_df.shape

(4180, 188)

### Calculating **Head-to-Head** Records

Determine the **head-to-head** performance against each opponent.

To do that I should again split the dataset into home and away matchs, because the calculation of the Head-to_Head metric is calculated different at the different perspectives!

First home team:

In [62]:
home_h2h = base_matches_df[
    [
        "datetime",
        "h_title",
        "a_title",
        "goal_diff",
        "xG_diff",
        "elo_rating_diff"
    ]
].copy()

home_h2h = home_h2h.rename(columns={
    "h_title": "team",
    "a_title": "opponent"
})

# Home perspective:
# positive goal_diff means HOME team performed better
home_h2h["h2h_goal_diff"] = home_h2h["goal_diff"]
home_h2h["h2h_xG_diff"] = home_h2h["xG_diff"]
home_h2h["h2h_elo_diff"] = home_h2h["elo_rating_diff"]

home_h2h["venue"] = "home"

Away team:

In [63]:
away_h2h = base_matches_df[
    [
        "datetime",
        "h_title",
        "a_title",
        "goal_diff",
        "xG_diff",
        "elo_rating_diff"
    ]
].copy()

away_h2h = away_h2h.rename(columns={
    "a_title": "team",
    "h_title": "opponent"
})

# Away perspective:
# Reversing the sign because: if home team won by +2, away team lost by -2.This is important to be considered
away_h2h["h2h_goal_diff"] = -away_h2h["goal_diff"]
away_h2h["h2h_xG_diff"] = -away_h2h["xG_diff"]
away_h2h["h2h_elo_diff"] = -away_h2h["elo_rating_diff"]

away_h2h["venue"] = "away"

Concating them into one dataset:

In [64]:
h2h_df = pd.concat(
    [home_h2h, away_h2h],
    ignore_index=True
)

In [65]:
h2h_df = h2h_df.sort_values(
    ['team', 'opponent', 'datetime']
).reset_index(drop=True)

#### Calculating Weighted Rolling Average:

Now this time instead of using the standard rolling average approach, I will use a function which creates rolling averages but for the averages, it applies different weights deppending on the match recency.The function applies exponentially increasing weights to the more recent matches, so that a more recent match to have a bigger impact than a more old one!This is a very good practice, because this way we capture the recent form of the teams, rather than relying on old results which might not be relevant anymore!

More about the function: [calculate_weighted_rolling_average](../../src/football_betting_analysis/features/features_creation.py)

Now lets use the functions and calculating the rolling averages for goal diff and expected goals diff:

In [66]:
h2h_df["h2h_rolling_goal_diff_5"] = (
    h2h_df.groupby(
        ["team", "opponent"],
        observed=False
    )["h2h_goal_diff"]
    .transform(lambda x: calculate_weighted_rolling_average(x, window=5))
)

h2h_df["h2h_rolling_xG_diff_5"] = (
    h2h_df.groupby(
        ["team", "opponent"],
        observed=False
    )["h2h_xG_diff"]
    .transform(lambda x: calculate_weighted_rolling_average(x, window=5))
)

h2h_df["h2h_rolling_elo_diff_5"] = (
    h2h_df.groupby(
        ["team", "opponent"],
        observed=False
    )["h2h_elo_diff"]
    .transform(lambda x: calculate_weighted_rolling_average(x, window=5))
)

Splitting the df into home and away matches, so to be merged with the base matches dataset:

In [67]:
home_h2h_features = (
    h2h_df[h2h_df['venue'] == 'home']
    .rename(columns={
        "team": "h_title",
        "opponent": "a_title",
        "h2h_rolling_goal_diff_5": "h_h2h_rolling_goal_diff_5",
        "h2h_rolling_xG_diff_5": "h_h2h_rolling_xG_diff_5",
        "h2h_rolling_elo_diff_5": "h_h2h_rolling_elo_diff_5"
    })
)

home_h2h_features = home_h2h_features[
    [
        "datetime",
        "h_title",
        "a_title",
        "h_h2h_rolling_goal_diff_5",
        "h_h2h_rolling_xG_diff_5",
        "h_h2h_rolling_elo_diff_5"
    ]
]

In [68]:
away_h2h_features = (
    h2h_df[h2h_df['venue'] == 'away']
    .rename(columns={
        "team": "a_title",
        "opponent": "h_title",
        "h2h_rolling_goal_diff_5": "a_h2h_rolling_goal_diff_5",
        "h2h_rolling_xG_diff_5": "a_h2h_rolling_xG_diff_5",
        "h2h_rolling_elo_diff_5": "a_h2h_rolling_elo_diff_5"
    })
)

away_h2h_features = away_h2h_features[
    [
        "datetime",
        "h_title",
        "a_title",
        "a_h2h_rolling_goal_diff_5",
        "a_h2h_rolling_xG_diff_5",
        "a_h2h_rolling_elo_diff_5"
    ]
]

Merge with the original base matches dataset:

In [69]:
base_matches_df = base_matches_df.merge(
    home_h2h_features,
    on=["datetime", "h_title", "a_title"],
    how="left"
)

base_matches_df = base_matches_df.merge(
    away_h2h_features,
    on=["datetime", "h_title", "a_title"],
    how="left"
)

### Optimizing the base matches dataset memory usage

In [70]:
base_matches_df = optimize_dataframe_memory(base_matches_df)

Initial Memory Usage: 4.13 MB
Final Memory Usage: 2.79 MB
Memory Reduction: 32.49%

Optimized Data Types:
league_code                        category
season                             category
datetime                     datetime64[us]
h_title                            category
a_title                            category
                                  ...      
h_h2h_rolling_xG_diff_5             float32
h_h2h_rolling_elo_diff_5            float32
a_h2h_rolling_goal_diff_5           float32
a_h2h_rolling_xG_diff_5             float32
a_h2h_rolling_elo_diff_5            float32
Length: 194, dtype: object


In [71]:
base_matches_df.shape

(4180, 194)

---

### Matches shots aggregation

Now I will use the Understat Matches Shots dataset to calculate different shots metrics. \
After the shots aggregation dataset is ready, it will be merged with the Base Matches Dataset and they together will form the **Final Dataset**!

In [72]:
understat_matches_shots_df.columns

Index(['match_id', 'season', 'datetime', 'minute', 'situation', 'X', 'Y',
       'shot_type', 'shot_result', 'last_action', 'xG', 'h_title', 'a_title',
       'player', 'player_id', 'h_a'],
      dtype='str')

In [73]:
shots_df = understat_matches_shots_df.copy()

### Removing columns:

In [74]:
shots_df = shots_df.drop(
    columns=['player', 'player_id']
)

### Adding new features:

#### Defining big chances:
I will start by creating a big chances features, which will simply indicate if the particular shot had a big chance to end up in a goal!Of course, this will be calculated using the expected goals(xG) metric!

To calculate this metric, it should be defined at which value of the expected goal, it could be considered a big chance!

#### Creating the big chance feature

> A shot is considered to be dangerous, when it has more then or equal to 35% probability of ending up in a goal, which means that the **xG**, should be more than or equal then 35%![1][1.2]

In [75]:
BIG_CHANCE_THRESHOLD = 0.35

In [76]:
shots_df["big_chance"] = (
    shots_df["xG"] >= BIG_CHANCE_THRESHOLD
).astype(int)

### Creating goal flag:

This feature is created so that to be used for measuring **shot conversion**!

In [77]:
shots_df["is_goal"] = (
    shots_df["shot_result"] == "Goal"
).astype(int)

### Splitting the shots into home and away ones:

In order to calculate shot features rightfully I should devide the shots data by home shots(the ones taken from home team) and away shots(the ones taken from the away team), and calculate the shot features for each of the teams!

In [78]:
home_shots = shots_df[
    shots_df["h_a"] == "h"
].copy()

away_shots = shots_df[
    shots_df["h_a"] == "a"
].copy()

### Aggreagting home shot features

First I will aggregate the home shots features:

In [79]:
home_shots['is_box_shot'] = home_shots['X'] >= 0.83
home_shots['is_high_xg'] = home_shots['xG'] >= 0.35
home_shots['open_play_xG'] = home_shots['xG'].where(home_shots['situation'] == 'OpenPlay', 0)
home_shots['set_piece_xG'] = home_shots['xG'].where(home_shots['situation'] != 'OpenPlay', 0)

In [80]:
home_features = (
    home_shots
    .groupby(['match_id', 'season', 'datetime', 'h_title', 'a_title'], observed=True)
    .agg(
        home_big_chances=("big_chance", "sum"),
        home_avg_xG_per_shot=("xG", "mean"),
        home_total_xG=("xG", "sum"),
        home_total_shots=("xG", "size"),
        home_shot_conversion=("is_goal", "mean"),
        home_avg_shot_distance=("X", "mean"),
        home_box_shots=("is_box_shot", "sum"),
        home_high_xg_shots=("is_high_xg", "sum"),
        home_open_play_xG=("open_play_xG", "sum"),
        home_set_piece_xG=("set_piece_xG", "sum")
    )
    .reset_index()
)

In [81]:
len(home_features) # Expecting to have 4180 total matches

4180

Now lets aggregate the away shot features:

In [82]:
away_shots['is_box_shot'] = away_shots['X'] >= 0.83
away_shots['is_high_xg'] = away_shots['xG'] >= 0.35
away_shots['open_play_xG'] = away_shots['xG'].where(away_shots['situation'] == 'OpenPlay', 0)
away_shots['set_piece_xG'] = away_shots['xG'].where(away_shots['situation'] != 'OpenPlay', 0)

In [83]:
away_features = (
    away_shots
    .groupby(['match_id', 'season', 'datetime', 'h_title', 'a_title'], observed=True)
    .agg(
        away_big_chances=("big_chance", "sum"),
        away_avg_xG_per_shot=("xG", "mean"),
        away_total_xG=("xG", "sum"),
        away_total_shots=("xG", "size"),
        away_shot_conversion=("is_goal", "mean"),
        away_avg_shot_distance=("X", "mean"),
        away_box_shots=("is_box_shot", "sum"),
        away_high_xg_shots=("is_high_xg", "sum"),
        away_open_play_xG=("open_play_xG", "sum"),
        away_set_piece_xG=("set_piece_xG", "sum")
    )
    .reset_index()
)

In [84]:
len(away_features) # Expecting to have 4180 total matches

4177

There are three missing matches, which will will be handled later.

### Merging the shot datasets:

As the features of the home and away shots are calculated, the datasets can be merged back together:

In [85]:
shot_features = home_features.merge(
    away_features,
    on=['match_id', 'datetime', 'h_title', 'a_title'],
    how='left',
    suffixes=["", "_a"]
)

In [86]:
shot_features = shot_features.drop(columns=['season_a'])

In [87]:
shot_features.columns

Index(['match_id', 'season', 'datetime', 'h_title', 'a_title',
       'home_big_chances', 'home_avg_xG_per_shot', 'home_total_xG',
       'home_total_shots', 'home_shot_conversion', 'home_avg_shot_distance',
       'home_box_shots', 'home_high_xg_shots', 'home_open_play_xG',
       'home_set_piece_xG', 'away_big_chances', 'away_avg_xG_per_shot',
       'away_total_xG', 'away_total_shots', 'away_shot_conversion',
       'away_avg_shot_distance', 'away_box_shots', 'away_high_xg_shots',
       'away_open_play_xG', 'away_set_piece_xG'],
      dtype='str')

### Creating rolling averages:

Calculating shots rolling averages:
- rolling_home_big_chances_5
- rolling_shot_conversion
- rolling_avg_xG_per_shot_5
- rolling_open_play_xG_5
- rolling_set_piece_xG_5

First I will devide the teams into home and away ones:

In [88]:
home_shots_df = shot_features[
    [
        'datetime', 
        'h_title', 
        'a_title', 
        'home_big_chances',
        'home_avg_xG_per_shot',
        'home_shot_conversion',
        'home_open_play_xG', 
        'home_set_piece_xG'
    ]
].rename(columns={
    "h_title": "team",
    "a_title": "opponent",
    'home_big_chances': 'big_chances',
    'home_avg_xG_per_shot': 'avg_xG_per_shot',
    'home_shot_conversion': 'shot_conversion',
    'home_open_play_xG': 'open_play_xG', 
    'home_set_piece_xG': 'set_piece_xG'
})

home_shots_df["venue"] = "home"

In [89]:
away_shots_df = shot_features[
    [
        'datetime',
        'a_title', 
        'h_title', 
        'away_big_chances',
        'away_avg_xG_per_shot',
        'away_shot_conversion',
        'away_open_play_xG', 
        'away_set_piece_xG'
    ]
].rename(columns={
    "a_title": "team",
    "h_title": "opponent",
    'away_big_chances': 'big_chances',
    'away_avg_xG_per_shot': 'avg_xG_per_shot',
    'away_shot_conversion': 'shot_conversion',
    'away_open_play_xG': 'open_play_xG', 
    'away_set_piece_xG': 'set_piece_xG'
})

away_shots_df["venue"] = "away"

In [90]:
# Merging the datasets
team_shots = pd.concat(
    [home_shots_df, away_shots_df],
    ignore_index=True
)

# Sorting the matches in chronological order
team_shots = team_shots.sort_values(
    ["team", "datetime", "venue"]
).reset_index(drop=True)

#### Calculating shots rolling averages:

In [91]:
shot_rolling_features = [
    'big_chances',
    'avg_xG_per_shot',
    'shot_conversion',
    'open_play_xG', 
    'set_piece_xG'
]

for feature in shot_rolling_features:
    rolling_col = f"rolling_{feature}_5"
    
    team_shots[rolling_col] = calculate_rolling_metric(
        team_shots,
        group_col='team',
        column=feature,
        func='mean',
        window=5
    )

In [92]:
team_shots.columns

Index(['datetime', 'team', 'opponent', 'big_chances', 'avg_xG_per_shot',
       'shot_conversion', 'open_play_xG', 'set_piece_xG', 'venue',
       'rolling_big_chances_5', 'rolling_avg_xG_per_shot_5',
       'rolling_shot_conversion_5', 'rolling_open_play_xG_5',
       'rolling_set_piece_xG_5'],
      dtype='str')

Now as we calculated the rolling averages for each of the teams(home/away), we should seperate them again:

In [93]:
home_shot_rolling = (
    team_shots[team_shots["venue"] == "home"]
    .rename(columns={
        "team": "h_title",
        'rolling_big_chances_5': 'h_rolling_big_chances_5', 
        'rolling_avg_xG_per_shot_5': 'h_rolling_avg_xG_per_shot_5',
        'rolling_shot_conversion_5': 'h_rolling_shot_conversion_5', 
        'rolling_open_play_xG_5': 'h_rolling_open_play_xG_5',
        'rolling_set_piece_xG_5': 'h_rolling_set_piece_xG_5'
    })
)

home_shot_rolling = home_shot_rolling[
    [
        "datetime",
        "h_title",
        'h_rolling_big_chances_5', 
        'h_rolling_avg_xG_per_shot_5',
        'h_rolling_shot_conversion_5', 
        'h_rolling_open_play_xG_5',
        'h_rolling_set_piece_xG_5'
    ]
]

Now for the away:

In [94]:
away_shot_rolling = (
    team_shots[team_shots["venue"] == "away"]
    .rename(columns={
        "team": "a_title",
        'rolling_big_chances_5': 'a_rolling_big_chances_5', 
        'rolling_avg_xG_per_shot_5': 'a_rolling_avg_xG_per_shot_5',
        'rolling_shot_conversion_5': 'a_rolling_shot_conversion_5', 
        'rolling_open_play_xG_5': 'a_rolling_open_play_xG_5',
        'rolling_set_piece_xG_5': 'a_rolling_set_piece_xG_5'
    })
)

away_shot_rolling = away_shot_rolling[
    [
        "datetime",
        "a_title",
        'a_rolling_big_chances_5', 
        'a_rolling_avg_xG_per_shot_5',
        'a_rolling_shot_conversion_5', 
        'a_rolling_open_play_xG_5',
        'a_rolling_set_piece_xG_5'
    ]
]

In [95]:
shot_features = shot_features.merge(
    home_shot_rolling,
    on=["datetime", "h_title"],
    how="left"
)

shot_features = shot_features.merge(
    away_shot_rolling,
    on=["datetime", "a_title"],
    how="left"
)

In [96]:
shot_features.columns

Index(['match_id', 'season', 'datetime', 'h_title', 'a_title',
       'home_big_chances', 'home_avg_xG_per_shot', 'home_total_xG',
       'home_total_shots', 'home_shot_conversion', 'home_avg_shot_distance',
       'home_box_shots', 'home_high_xg_shots', 'home_open_play_xG',
       'home_set_piece_xG', 'away_big_chances', 'away_avg_xG_per_shot',
       'away_total_xG', 'away_total_shots', 'away_shot_conversion',
       'away_avg_shot_distance', 'away_box_shots', 'away_high_xg_shots',
       'away_open_play_xG', 'away_set_piece_xG', 'h_rolling_big_chances_5',
       'h_rolling_avg_xG_per_shot_5', 'h_rolling_shot_conversion_5',
       'h_rolling_open_play_xG_5', 'h_rolling_set_piece_xG_5',
       'a_rolling_big_chances_5', 'a_rolling_avg_xG_per_shot_5',
       'a_rolling_shot_conversion_5', 'a_rolling_open_play_xG_5',
       'a_rolling_set_piece_xG_5'],
      dtype='str')

#### Now the dataset contains rolling shots metrcis!

Actually, the reason why the rolling features are so important, is because they provide the teams performances before the actual match, without including any current match stats, because otherwise, without these rolling features, the matches only containts stats from the matches themselves, which cannot be used in predictions, because it is expected that the matches stats are unknown yet, so the things that should be used are precisely the rolling features, because they provide performance before the actual matches!!!

### Removing features from the shots dataset

Some features are useless and should be removed:

In [97]:
shot_features = shot_features.drop(
    columns=[
        'match_id',
        'home_total_xG',
        'away_total_xG',
        'home_total_shots',
        'away_total_shots'
    ]
)

In [98]:
shot_features.columns

Index(['season', 'datetime', 'h_title', 'a_title', 'home_big_chances',
       'home_avg_xG_per_shot', 'home_shot_conversion',
       'home_avg_shot_distance', 'home_box_shots', 'home_high_xg_shots',
       'home_open_play_xG', 'home_set_piece_xG', 'away_big_chances',
       'away_avg_xG_per_shot', 'away_shot_conversion',
       'away_avg_shot_distance', 'away_box_shots', 'away_high_xg_shots',
       'away_open_play_xG', 'away_set_piece_xG', 'h_rolling_big_chances_5',
       'h_rolling_avg_xG_per_shot_5', 'h_rolling_shot_conversion_5',
       'h_rolling_open_play_xG_5', 'h_rolling_set_piece_xG_5',
       'a_rolling_big_chances_5', 'a_rolling_avg_xG_per_shot_5',
       'a_rolling_shot_conversion_5', 'a_rolling_open_play_xG_5',
       'a_rolling_set_piece_xG_5'],
      dtype='str')

### Fixing data types

Some of the columns are with unappropriate data types, so they should be fixed:

In [99]:
shot_features = validate_and_cast_dataframe_dtypes(shot_features)

Ok now every column is with appropriate data type!

### Optimizing dataset memory usage:

Most of the columns can be casted to more small-sized types, which will reduce a lot of memory!

In [100]:
shot_features = optimize_dataframe_memory(shot_features)

Initial Memory Usage: 0.76 MB
Final Memory Usage: 0.48 MB
Memory Reduction: 36.25%

Optimized Data Types:
season                               category
datetime                       datetime64[us]
h_title                              category
a_title                              category
home_big_chances                         int8
home_avg_xG_per_shot                  float32
home_shot_conversion                  float32
home_avg_shot_distance                float32
home_box_shots                           int8
home_high_xg_shots                       int8
home_open_play_xG                     float32
home_set_piece_xG                     float32
away_big_chances                        Int64
away_avg_xG_per_shot                  float32
away_shot_conversion                  float32
away_avg_shot_distance                float32
away_box_shots                          Int64
away_high_xg_shots                      Int64
away_open_play_xG                     float32
away_set_piece_xG   

### Ordering the shot features dataset

In [101]:
shot_features = shot_features.sort_values('datetime').reset_index(drop=True)

## Creating the final dataset:

Now as the shots features have been created, and all of the components are cleaned, it's time to **merge the Base Matches dataset and the Shot Features dataset**!

But before merging I should be ensure that there are no duplicated matches across the datasets:

In [102]:
shot_features.duplicated(subset=['datetime', 'h_title', 'a_title']).sum()

np.int64(0)

In [103]:
base_matches_df.duplicated(subset=['datetime', 'h_title', 'a_title']).sum()

np.int64(0)

#### Another thing which is almost certain, is that, as the first **understat matches dataset** was with wrong matches dates(one day ahead of the real date), this shot fetaures dataset will also be with invalid dates, because it is created from the **Understat matches shots** dataset!

### Resolving the **ShotFeatures** dataset matches dates:

In [104]:
base_matches_df = base_matches_df.copy()
shot_features = shot_features.copy()

base_matches_df["base_idx"] = base_matches_df.index
shot_features["shot_idx"] = shot_features.index

In [105]:
test_df = base_matches_df.merge(
    shot_features,
    on=[
        "season",
        "datetime",
        "h_title",
        "a_title"
    ],
    how="outer",
    indicator=True
)

In [106]:
test_df.shape

(4250, 223)

Yes they are certainly more than they should be, so this dataset should also be fixed as the first Understat matches one.

Getting the ones which differ:

In [107]:
left_only = test_df[test_df["_merge"] == "left_only"].copy()
right_only = test_df[test_df["_merge"] == "right_only"].copy()

Now lets apply the **resolve_date_shift_matches** function, which will fix the problem:

In [108]:
resolved_matches = resolve_date_shift_matches(
    left_df=left_only,
    right_df=right_only,
    use_index=True,
    left_index_col='base_idx',
    right_index_col='shot_idx',
    max_days=1
)

In [109]:
resolved_matches

,left_index,right_index
0,384.0,384.0
1,387.0,388.0
2,386.0,387.0
3,392.0,396.0
4,393.0,394.0
...,...,...
65,1224.0,1229.0
66,1242.0,1244.0
67,3020.0,3022.0
68,3182.0,3180.0


Now lets use the resolved matches that we have craeted and fix the **ShotFeature dataset**:

In [110]:
for _, row in resolved_matches.iterrows():

    left_idx = row["left_index"]
    right_idx = row["right_index"]

    corrected_date = base_matches_df.loc[left_idx, "datetime"]

    shot_features.loc[right_idx, "datetime"] = corrected_date

Now lets remove the helping index columns:

In [111]:
base_matches_df = base_matches_df.drop(columns=['base_idx'])
shot_features = shot_features.drop(columns=['shot_idx'])

In [112]:
final_df = base_matches_df.merge(
    shot_features,
    on=[
        "season",
        "datetime",
        "h_title",
        "a_title"
    ],
    how="inner",
)

In [113]:
final_df.shape

(4180, 220)

Ahh, now everything seems to be perfect:

In [114]:
final_df.columns.tolist()

['league_code',
 'season',
 'datetime',
 'h_title',
 'a_title',
 'home_elo',
 'away_elo',
 'home_form_last_3',
 'home_form_last_5',
 'away_form_last_3',
 'away_form_last_5',
 'home_goals_full',
 'away_goals_full',
 'result_full',
 'home_goals_half',
 'away_goals_half',
 'result_half',
 'home_shots',
 'away_shots',
 'home_shots_on_target',
 'away_shots_on_target',
 'home_fouls',
 'away_fouls',
 'home_corners',
 'away_corners',
 'home_yellow_cards',
 'away_yellow_cards',
 'home_red_cards',
 'away_red_cards',
 'handicap_size',
 'handicap_home',
 'handicap_away',
 'xG_h',
 'xG_a',
 'odds_bet365_home',
 'odds_bet365_draw',
 'odds_bet365_away',
 'time',
 'h_poss',
 'a_poss',
 'referee',
 'round',
 'game_id',
 'home_club_id',
 'away_club_id',
 'home_club_position',
 'away_club_position',
 'home_club_manager_name',
 'away_club_manager_name',
 'stadium',
 'attendance',
 'home_missing_players_count',
 'home_missing_key_players_count',
 'home_missing_star_players_count',
 'home_missing_importance

### Creating external features:

Now I will continue with the creation of more external type features.Most of the times, the biggest impact and influence do not comes from the game stats themselves, but from some external factors, which despite of being hidden and unseen, they drive many of the games and their outcomes!

Here are some of these external factors which I plan to create:
1. Formation stability & tactical consistency
2. Manager stability & regime coherence
3. Referee bias (minimal, high-signal features only)
4. Attendance pressure amplifier

##### About the formation and manager stability:
I have implemented the `compute_formation_stability_features` and `compute_manager_features`, which simply look back at history of the teams(both home and away) and observes the what formations and managers do the teams have and do they tend to have a consistent and stable formation and a manager.These factors are very important, because if we take a team which always switches his formations and changes its manager(the one leeds to the other), this team surely cannot find his comfort and therefore, a reliable manager which can form the right formation and use the team and its strengths as much as possible!

##### About the referee bias
I have implemented the function `compute_referee_bias_features` which computes the average made fouls, cards and home wins for all of the matches and then it compares them with the fould, card and home wins of the specific referees for each match.The aim of the function is to create features which identifies if a certain referee tends to judge more fouls or maybe to have a home team bias.**Such features can be extremely important because most of the times a match is not won by the teams but by the referees!**

##### About the Attendance pressure amplifier:
I have implemented a function `compute_attendance_features` which computes the average team's attendace for the matches and compares it to the their current match atteandance.If a certain match has a lot of attendance it is almost certain that the match is very important for the teams.So such things can be used to identify for a team presure for example and also the pressure from the attendance can be combined with other features like the importance of the match for the teams and this way to form a really solid team pressure index!

Enough explanaition, lets use the functions and create the final features!
### Computing formations stability features:

In [115]:
# I need to rename this columns because the functions expect the column to be named: 'date' and the exact time column: 'time'
final_df = final_df.rename(columns={'datetime': 'date'})

In [116]:
final_df = compute_formation_stability_features(final_df)

### Computing manager stability features: 

In [117]:
final_df = compute_manager_features(final_df)

### Computing referee bias features:

In [118]:
final_df = compute_referee_bias_features(final_df)

### Computing attendance pressure features:

In [119]:
final_df = compute_attendance_features(final_df)

In [120]:
final_df.shape

(4180, 239)

## Removing features

Now I will remove some of the columns which in one or another way are useless and cannot be used in any meaningful way!Currently the dataset contains a lot of in-match features which cannot be used for the predictions, just because they present data from the matches themself and if I leave it, this would mean to introduce data leakage!

So lets remove all of the features that present info from the matches:

In [121]:
final_df = final_df.drop(
    columns=[
        'league_code',
        'home_goals_full',
        'away_goals_full',
        'home_goals_half',
        'away_goals_half',
        'result_half',
        'handicap_size',
        'home_shots',
        'away_shots',
        'home_shots_on_target',
        'away_shots_on_target',
        'home_fouls',
        'away_fouls',
        'home_corners',
        'away_corners',
        'home_yellow_cards',
        'away_yellow_cards',
        'home_red_cards',
        'away_red_cards',
        'xG_h',
        'xG_a',
        'h_poss',
        'a_poss',
        'goal_diff',
        'xG_diff',
        'poss_diff',
        'home_big_chances',
        'home_avg_xG_per_shot',
        'home_shot_conversion',
        'home_avg_shot_distance',
        'home_box_shots',
        'home_high_xg_shots',
        'home_open_play_xG',
        'home_set_piece_xG',
        'away_big_chances',
        'away_avg_xG_per_shot',
        'away_shot_conversion',
        'away_avg_shot_distance',
        'away_box_shots',
        'away_high_xg_shots',
        'away_open_play_xG',
        'away_set_piece_xG',
    ]
)

In [122]:
final_df.shape

(4180, 197)

In [123]:
final_df.columns.tolist()

['season',
 'date',
 'h_title',
 'a_title',
 'home_elo',
 'away_elo',
 'home_form_last_3',
 'home_form_last_5',
 'away_form_last_3',
 'away_form_last_5',
 'result_full',
 'handicap_home',
 'handicap_away',
 'odds_bet365_home',
 'odds_bet365_draw',
 'odds_bet365_away',
 'time',
 'referee',
 'round',
 'game_id',
 'home_club_id',
 'away_club_id',
 'home_club_position',
 'away_club_position',
 'home_club_manager_name',
 'away_club_manager_name',
 'stadium',
 'attendance',
 'home_missing_players_count',
 'home_missing_key_players_count',
 'home_missing_star_players_count',
 'home_missing_importance_sum',
 'home_missing_defenders',
 'home_missing_midfielders',
 'home_missing_forwards',
 'home_starting_goalkeeper_missing',
 'home_missing_captain',
 'home_missing_top1_player',
 'home_missing_top3_player',
 'home_available_strength',
 'home_missing_expected_starter_strength',
 'home_missing_gk_strength',
 'home_missing_def_strength',
 'home_missing_mid_strength',
 'home_missing_fwd_strength',
 

---

## Handling missing values:

The last problem which should be handled are the missing values.Most of the values are missing because of the lack of history data from the first matches of the teams, and this is logical.But I cannot leave so many missing values!

Lets actually, see which of the columns have missing values: 

In [124]:
missing_data = pd.DataFrame({
    'Missing Count': final_df.isna().sum(),
    'Percentage (%)': (final_df.isna().sum() / len(final_df)) * 100
})

# Filter out columns that have 0 missing values
missing_data[missing_data['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

,Missing Count,Percentage (%)
a_h2h_rolling_goal_diff_5,432,10.334928
h_h2h_rolling_elo_diff_5,432,10.334928
h_h2h_rolling_xG_diff_5,432,10.334928
h_h2h_rolling_goal_diff_5,432,10.334928
a_h2h_rolling_elo_diff_5,432,10.334928
...,...,...
home_missing_def_strength,2,0.047847
home_missing_importance_ratio,2,0.047847
home_missing_mid_strength,2,0.047847
home_missing_fwd_strength,2,0.047847


In [125]:
missing_data['Missing Count'].value_counts().sort_index(ascending=False)

Missing Count
432      6
16      17
15      17
3       19
2       19
0      119
Name: count, dtype: int64

As we can see only six of the columns have a lot missing values.The other missing data is either a lack of history data or a problem in the functions.

#### However, for now I won't do anything with these missing data.They will be handled into the data preprocessing phrase!I will first insepect the missing values a little bit more detailly in the eda project part and then I will impute them professionally in the data preprocessing part, where also most of the features will be removed and the data will prepared for modelling!

Now lets optimize the final dataset and save it:

### Optimizing the dataset memory usage:

In [126]:
final_df = optimize_dataframe_memory(final_df)

Initial Memory Usage: 3.31 MB
Final Memory Usage: 2.91 MB
Memory Reduction: 12.17%

Optimized Data Types:
season                          category
date                      datetime64[us]
h_title                         category
a_title                         category
home_elo                         float32
                               ...      
referee_card_rate                float32
attendance_num                   float32
attendance_ratio                 float32
high_crowd_pressure                 int8
home_attendance_vs_avg           float32
Length: 197, dtype: object


In [127]:
final_df.shape

(4180, 197)

## Saving the final dataset

Saving the final dataset into the processed data folder!

In [128]:
# force working directory to project root
PROJECT_ROOT = Path().resolve().parent.parent

In [129]:
processed_final_dataset_path = PROJECT_ROOT / PROCESSED_DATA_PATH / 'processed_final_dataset_v1.parquet'

save_data(data=final_df, file_path=processed_final_dataset_path)

The data has successfully been saved into: C:\Users\Asus\Desktop\Data_Science\Football_Betting_Analysis\Football_Betting_Analysis\data\processed\processed_final_dataset_v1.parquet
